In [2]:
%load_ext sql

In [3]:
%sql spark

In [4]:
%%sql
show tables in spark_catalog.deltalake;

Running query in 'SparkSession'

12 rows affected.

Field 1,Field 2,Field 3
deltalake,delta_table,False
deltalake,invoices_cow,False


In [65]:
%%sql

DROP TABLE IF EXISTS spark_catalog.deltalake.optimize_ex1;
DROP TABLE IF EXISTS spark_catalog.deltalake.optimize_ex2;
DROP TABLE IF EXISTS spark_catalog.deltalake.optimize_ex3;

Running query in 'SparkSession'

++
||
++
++

In [46]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/optimize_ex1 --recursive

In [47]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/optimize_ex2 --recursive

In [48]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/optimize_ex3 --recursive

#### OPTIMIZE Intro

In [6]:
df = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_201_99457.parquet")
display(df.limit(5))

+-----------+----------+------+---+--------+--------+------+--------------+------------+-----------------+-------------+
|customer_id|invoice_no|gender|age|category|quantity|price |payment_method|invoice_date|shopping_mall    |_rescued_data|
+-----------+----------+------+---+--------+--------+------+--------------+------------+-----------------+-------------+
|201        |I885979   |Female|26 |Clothing|3       |900.24|Debit Card    |2021-07-04  |Metrocity        |NULL         |
|202        |I810217   |Female|51 |Clothing|3       |900.24|Cash          |2022-01-14  |Metrocity        |NULL         |
|203        |I499170   |Female|38 |Toys    |1       |35.84 |Credit Card   |2022-02-20  |Kanyon           |NULL         |
|204        |I792963   |Female|59 |Clothing|5       |1500.4|Debit Card    |2022-06-18  |Emaar Square Mall|NULL         |
|205        |I311151   |Female|39 |Souvenir|3       |35.19 |Credit Card   |2022-04-27  |Mall of Istanbul |NULL         |
+-----------+----------+------+-

In [7]:
print(df.count())
print(df.select("category").distinct().count())

99257
8


In [8]:
#
df.repartition(5).write.format("delta").mode("overwrite").partitionBy("category").saveAsTable("spark_catalog.deltalake.optimize_ex1")


In [9]:
%%time

df_ex1 = spark.read.format("delta").table("spark_catalog.deltalake.optimize_ex1")
df_out = df_ex1.where(df_ex1.category == "Clothing").collect()


CPU times: user 183 ms, sys: 12 ms, total: 195 ms
Wall time: 1.76 s


In [10]:
from delta.tables import *

deltaTable = DeltaTable.forName(spark, "deltalake.optimize_ex1")
deltaTable.optimize().executeCompaction()


DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [12]:
deltaTable.vacuum(0)

IllegalArgumentException: requirement failed: Are you sure you would like to vacuum files with such a low retention period? If you have
writers that are currently writing to this table, there is a risk that you may corrupt the
state of your Delta table.

If you are certain that there are no operations being performed on this table, such as
insert/upsert/delete/optimize, then you may turn off this check by setting:
spark.databricks.delta.retentionDurationCheck.enabled = false

If you are not sure, please use a value not less than "168 hours".
       

In [13]:
%%sql
 
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
spark.databricks.delta.retentionDurationCheck.enabled,false


In [14]:
%%time

df_ex1 = spark.read.format("delta").table("spark_catalog.deltalake.optimize_ex1")
df_out = df_ex1.where(df_ex1.category == "Clothing").collect()

CPU times: user 216 ms, sys: 7.99 ms, total: 224 ms
Wall time: 1.2 s


In [15]:
import pyspark.sql.functions as F
df_fruits = df.filter(df.category == "Clothing").withColumn("category", F.lit("Fruits"))

In [16]:
df_fruits.select("category").distinct().count()

1

In [17]:
df_fruits.repartition(5).write.format("delta").mode("overwrite").partitionBy("category").saveAsTable("spark_catalog.deltalake.optimize_ex1")

26/04/01 17:01:49 ERROR HiveAlterHandler: Failed to alter table deltalake.optimize_ex1


In [0]:
# table.optimize().where("category = 'Fruits'").executeCompaction()

In [18]:
%%sql

OPTIMIZE spark_catalog.deltalake.optimize_ex1
WHERE category = 'Fruits';

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
s3a://warehouse/default/deltalake/optimize_ex1,"Row(numFilesAdded=1, numFilesRemoved=5, filesAdded=Row(min=480214, max=480214, avg=480214.0, totalFiles=1, totalSize=480214), filesRemoved=Row(min=101457, max=101617, avg=101536.8, totalFiles=5, totalSize=507684), partitionsOptimized=1, zOrderStats=None, clusteringStats=None, numBins=1, numBatches=1, totalConsideredFiles=5, totalFilesSkipped=0, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1775043118373, endTimeMs=0, totalClusterParallelism=8, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=None, numTableColumns=11, numTableColumnsWithStats=11)"


In [19]:
%%sql

VACUUM spark_catalog.deltalake.optimize_ex1 RETAIN 0 HOURS;

Running query in 'SparkSession'

Deleted 61 files and directories in a total of 1 directories.


1 rows affected.

Field 1
s3a://warehouse/default/deltalake/optimize_ex1


#### Optimized Write

In [20]:
#
df.repartition(288).write.format("delta").mode("overwrite").partitionBy("category").saveAsTable("spark_catalog.deltalake.optimize_ex2")

In [21]:
%%time

df_ex2 = spark.read.format("delta").table("spark_catalog.deltalake.optimize_ex2")
df_out = df_ex2.where(df_ex2.category == "Clothing").collect()

CPU times: user 75.6 ms, sys: 10.4 ms, total: 86.1 ms
Wall time: 1.8 s


In [22]:
#
df.repartition(288).write.format("delta").mode("overwrite").partitionBy("category").option("optimizeWrite", "True").saveAsTable("spark_catalog.deltalake.optimize_ex3")

In [23]:
%%time

df_ex3 = spark.read.format("delta").table("spark_catalog.deltalake.optimize_ex3")
df_out = df_ex3.where(df_ex3.category == "Clothing").collect()

CPU times: user 78.4 ms, sys: 4.76 ms, total: 83.1 ms
Wall time: 803 ms


#### Auto Compaction

In [24]:
%%sql
 
ALTER TABLE spark_catalog.deltalake.optimize_ex3 SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'false');
ALTER TABLE spark_catalog.deltalake.optimize_ex3 SET TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'true');

Running query in 'SparkSession'

++
||
++
++

In [25]:

spark.conf.get("spark.databricks.delta.autoCompact.minNumFiles")
# spark.conf.set("spark.databricks.delta.autoCompact.minNumFiles", "3")

'50'

In [26]:
df_detergents = df.filter(df.category == "Clothing").withColumn("category", F.lit("Detergents"))

In [27]:
df_detergents.repartition(5).write.format("delta").mode("overwrite").partitionBy("category").saveAsTable("spark_catalog.deltalake.optimize_ex3")

26/04/01 17:03:34 ERROR HiveAlterHandler: Failed to alter table deltalake.optimize_ex3


#### VACUUM 
(How it limits ability to time travel)

In [28]:
df_101_150 = (
    spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet")
    .filter(F.col("customer_id").between(101, 150))
)

df_151_200 = (
    spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet")
    .filter(F.col("customer_id").between(151, 200))
)

In [29]:
df_101_150.write.format("delta").mode("overwrite").saveAsTable("spark_catalog.deltalake.vacuum_ex1")

26/04/01 17:03:46 ERROR HiveAlterHandler: Failed to alter table deltalake.vacuum_ex1


In [30]:
%%sql

DESCRIBE HISTORY spark_catalog.deltalake.vacuum_ex1;

Running query in 'SparkSession'

2 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
1,2026-04-01 17:03:45,None,None,CREATE OR REPLACE TABLE AS SELECT,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,0,Serializable,False,"{'numOutputRows': '50', 'numOutputBytes': '4683', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-04-01 16:17:56,None,None,CREATE OR REPLACE TABLE AS SELECT,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,None,Serializable,False,"{'numOutputRows': '50', 'numOutputBytes': '4683', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [31]:
df_151_200.write.format("delta").mode("append").saveAsTable("spark_catalog.deltalake.vacuum_ex1")

In [32]:
%%sql

DESCRIBE HISTORY spark_catalog.deltalake.vacuum_ex1;

Running query in 'SparkSession'

3 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
2,2026-04-01 17:04:18,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,1,Serializable,True,"{'numOutputRows': '50', 'numOutputBytes': '4728', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
1,2026-04-01 17:03:45,None,None,CREATE OR REPLACE TABLE AS SELECT,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,0,Serializable,False,"{'numOutputRows': '50', 'numOutputBytes': '4683', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-04-01 16:17:56,None,None,CREATE OR REPLACE TABLE AS SELECT,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,None,Serializable,False,"{'numOutputRows': '50', 'numOutputBytes': '4683', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [33]:
%%sql

DELETE FROM spark_catalog.deltalake.vacuum_ex1 WHERE customer_id BETWEEN 151 AND 200; 

Running query in 'SparkSession'

1 rows affected.

Field 1
50


In [34]:
%%sql

DESCRIBE HISTORY spark_catalog.deltalake.vacuum_ex1;

Running query in 'SparkSession'

4 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
3,2026-04-01 17:04:36,None,None,DELETE,"{'predicate': '[""((customer_id#26142 >= 151) AND (customer_id#26142 <= 200))""]'}",None,None,None,2,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '0', 'executionTimeMs': '1021', 'numDeletionVectorsRemoved': '0', 'numRemovedFiles': '1', 'rewriteTimeMs': '118', 'numRemovedBytes': '4728', 'scanTimeMs': '903', 'numCopiedRows': '0', 'numDeletionVectorsAdded': '0', 'numAddedChangeFiles': '0', 'numDeletedRows': '50', 'numAddedBytes': '0'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
2,2026-04-01 17:04:18,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,1,Serializable,True,"{'numOutputRows': '50', 'numOutputBytes': '4728', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
1,2026-04-01 17:03:45,None,None,CREATE OR REPLACE TABLE AS SELECT,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,0,Serializable,False,"{'numOutputRows': '50', 'numOutputBytes': '4683', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-04-01 16:17:56,None,None,CREATE OR REPLACE TABLE AS SELECT,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'true'}",None,None,None,None,Serializable,False,"{'numOutputRows': '50', 'numOutputBytes': '4683', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [35]:
%%sql
 
SELECT MIN(customer_id), MAX(customer_id)
FROM spark_catalog.deltalake.vacuum_ex1;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
101,150


In [36]:
%%sql
 
SELECT MIN(customer_id), MAX(customer_id)
FROM spark_catalog.deltalake.vacuum_ex1 VERSION AS OF 1;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
101,150


In [37]:
%%sql

SET spark.databricks.delta.retentionDurationCheck.enabled = false;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
spark.databricks.delta.retentionDurationCheck.enabled,false


In [38]:
%%sql

VACUUM spark_catalog.deltalake.vacuum_ex1 RETAIN 0 HOURS;

Running query in 'SparkSession'

Deleted 12 files and directories in a total of 1 directories.


1 rows affected.

Field 1
s3a://warehouse/default/deltalake/vacuum_ex1


In [39]:
%%sql

DESCRIBE HISTORY spark_catalog.deltalake.vacuum_ex1;

Running query in 'SparkSession'

RuntimeError: [DELTA_TABLE_ONLY_OPERATION] `s3a://warehouse/default/deltalake/vacuum_ex1` is not a Delta table. DESCRIBE HISTORY is only supported for Delta tables.


In [40]:
%%sql

SELECT * 
FROM spark_catalog.deltalake.vacuum_ex1 VERSION AS OF 0;

Running query in 'SparkSession'

RuntimeError: [DELTA_NO_COMMITS_FOUND] No commits found at s3a://warehouse/default/deltalake/vacuum_ex1/_delta_log
